## Case When formatting test

### Goals

Work in progress -- ignore this for now


### Set Up -- Tree Code

In [1]:
import duckdb
import polars as pl
import polars.selectors as cs
import sqlglot
import pandas as pd
import numpy as np

In [2]:
# nested case when version
d = []
for i in np.arange(8):
    d += [ f"case when d{i+1} > 0.5 then {i+1} else 0 end"]

c = []
for i in np.arange(4):
    sql_temp = f"case when c{i+1} > 0.5 then {d[2*i]} else {d[2*i+1]} end"
    c += [sql_temp]

b = []
for i in np.arange(2):
    sql_temp = f"case when b{i+1} > 0.5 then {c[2*i]} else {c[2*i+1]} end"
    b += [sql_temp]

sql_nest = f"case when a > 0.5 then {b[0]} else {b[1]} end"

print(
    sqlglot.transpile(sql_nest, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5
  THEN CASE
    WHEN "b1" > 0.5
    THEN CASE
      WHEN "c1" > 0.5
      THEN CASE WHEN "d1" > 0.5 THEN 1 ELSE 0 END
      ELSE CASE WHEN "d2" > 0.5 THEN 2 ELSE 0 END
    END
    ELSE CASE
      WHEN "c2" > 0.5
      THEN CASE WHEN "d3" > 0.5 THEN 3 ELSE 0 END
      ELSE CASE WHEN "d4" > 0.5 THEN 4 ELSE 0 END
    END
  END
  ELSE CASE
    WHEN "b2" > 0.5
    THEN CASE
      WHEN "c3" > 0.5
      THEN CASE WHEN "d5" > 0.5 THEN 5 ELSE 0 END
      ELSE CASE WHEN "d6" > 0.5 THEN 6 ELSE 0 END
    END
    ELSE CASE
      WHEN "c4" > 0.5
      THEN CASE WHEN "d7" > 0.5 THEN 7 ELSE 0 END
      ELSE CASE WHEN "d8" > 0.5 THEN 8 ELSE 0 END
    END
  END
END
CASE
  WHEN "a" > 0.5
  THEN CASE
    WHEN "b1" > 0.5
    THEN CASE
      WHEN "c1" > 0.5
      THEN CASE WHEN "d1" > 0.5 THEN 1 ELSE 0 END
      ELSE CASE WHEN "d2" > 0.5 THEN 2 ELSE 0 END
    END
    ELSE CASE
      WHEN "c2" > 0.5
      THEN CASE WHEN "d3" > 0.5 THEN 3 ELSE 0 END
      ELSE CASE WHEN "d4" > 0.5 THEN 4

In [3]:
# linearize case when version 
# using strict versus equalities versus inclusion / exclusion just so my code lines up nice =) 
# it's a timing exercise so it really doesn't matter
sql_line = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 < 0.5 then 0 
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 < 0.5 then 0
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 < 0.5 then 0 
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 > 0.5 then 4
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 < 0.5 then 0
-- a <= 0.5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 < 0.5 then 0 
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 > 0.5 then 6
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 < 0.5 then 0
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 > 0.5 then 7
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 < 0.5 then 0 
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 > 0.5 then 8
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 < 0.5 then 0
else null end
'''

print(
    sqlglot.transpile(sql_line, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" < 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" < 0.5 AND "d2" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" > 0.5 AND "d3" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" < 0.5 AND "d4" > 0.5
  THEN 4
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" < 0.5 AND "d4" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" > 0.5 AND "d5" > 0.5
  THEN 5
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" > 0.5 AND "d5" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" < 0.5 AND "d6" > 0.5
  THEN 6
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" < 0.5 AND "d6" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" < 0.5 AND "c4" > 0.5 AND "d7" > 0.5
  THEN 7
  WHEN "a" < 0.5 AND "b2" < 0.5 AND "c4" > 0.5 AND "d7" < 0.5
  THEN 0
 

In [4]:
# linearized case when version -- pruning redundant conditions
sql_slim = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5              then 0 
when a > 0.5 and b1 > 0.5              and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5                           then 0
when a > 0.5 and              c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and              c2 > 0.5              then 0 
when a > 0.5 and                           d4 > 0.5 then 4
when a > 0.5                                        then 0
-- a <= 0.5
when             b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when             b2 > 0.5 and c3 > 0.5              then 0 
when             b2 > 0.5              and d6 > 0.5 then 6
when             b2 > 0.5                           then 0
when                          c4 > 0.5 and d7 > 0.5 then 7
when                          c4 > 0.5              then 0 
when                                       d8 > 0.5 then 8
when a < 0.5                                        then 0
else null end
'''

print(
    sqlglot.transpile(sql_slim, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5 AND "c2" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "d4" > 0.5
  THEN 4
  WHEN "a" > 0.5
  THEN 0
  WHEN "b2" > 0.5 AND "c3" > 0.5 AND "d5" > 0.5
  THEN 5
  WHEN "b2" > 0.5 AND "c3" > 0.5
  THEN 0
  WHEN "b2" > 0.5 AND "d6" > 0.5
  THEN 6
  WHEN "b2" > 0.5
  THEN 0
  WHEN "c4" > 0.5 AND "d7" > 0.5
  THEN 7
  WHEN "c4" > 0.5
  THEN 0
  WHEN "d8" > 0.5
  THEN 8
  WHEN "a" < 0.5
  THEN 0
  ELSE NULL
END
CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5

### Set Up -- Data

In [5]:
# set up random data matrix
n = 1000000
p = 15
df = pl.DataFrame( np.random.rand(n,p) )
df.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']
df.glimpse()

Rows: 1000000
Columns: 15
$ a  <f64> 0.656329440486069, 0.6530512469143721, 0.44518586261429105, 0.26268048422803036, 0.07412285639785121, 0.817646914362074, 0.22589958388080122, 0.7959263357583884, 0.825954594292577, 0.11476114950188276
$ b1 <f64> 0.5690058205217744, 0.6300542197959924, 0.365778678295421, 0.934295042491303, 0.5472310436568669, 0.5171604870943063, 0.3405254745795241, 0.2347598388022022, 0.47546390864635957, 0.8161189332700377
$ b2 <f64> 0.9971075946913132, 0.08397905495117641, 0.37449396637907517, 0.5430097936493858, 0.8225796319322444, 0.007289686944786267, 0.1498014245330055, 0.968938411210766, 0.09992143470571813, 0.8457984724388636
$ c1 <f64> 0.2956495060601806, 0.6320925193699666, 0.48131122528752435, 0.5837576110184101, 0.39966077734026384, 0.568460303334582, 0.8093424687676333, 0.45217600227888, 0.25298015529639184, 0.3319428400495763
$ c2 <f64> 0.561654594595016, 0.7726685463985447, 0.6702098711893841, 0.4073061170409151, 0.7078322889614775, 0.7731487288345992,

In [6]:
# ensure different candidates have same logic
sql_compare = f"""
select
{sql_nest} as out_nest,
{sql_line} as out_line,
{sql_slim} as out_slim,
*
from df
"""
df_out = duckdb.sql(sql_compare).pl()
df_out.filter( 
    (pl.col('out_line') != pl.col('out_slim')) |
    (pl.col('out_nest') != pl.col('out_slim'))
)

out_nest,out_line,out_slim,a,b1,b2,c1,c2,c3,c4,d1,d2,d3,d4,d5,d6,d7,d8
i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [7]:
# base case output frequency
df_out.group_by('out_nest').len()

out_nest,len
i32,u32
1,62335
8,62430
7,62628
2,62511
3,62227
6,62387
0,500757
5,62383
4,62342


In [8]:
# create different skews
df_skew = (
df_out
.with_columns(threshhold = pl.when(pl.col('out_nest') > 0).then(10 - pl.col('out_nest')).otherwise(5))
.filter( pl.col('out_nest').cum_count().over('out_nest') <= pl.col('threshhold')*7000 )
)

print(df_skew.shape[0])

(
df_skew
.group_by('out_nest')
.len()
.sort('out_nest')
.with_columns( p = pl.col('len') / pl.col('len').sum() )
)

342335
342335


out_nest,len,p
i32,u32,f64
0,35000,0.102239
1,62335,0.182088
2,56000,0.163582
3,49000,0.143135
4,42000,0.122687
5,35000,0.102239
6,28000,0.081791
7,21000,0.061343
8,14000,0.040896


In [9]:
# final prep - standardizing data size
df_skew = pl.concat([df_skew]*4).drop( cs.starts_with('out_') )
df_unif = pl.DataFrame( np.random.rand( df_skew.shape[0],p) )
df_unif.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']

In [10]:
### Timing

con = duckdb.connect()
con.sql("SET enable_object_cache = false;")

#### Base Case

In this case all nodes are equally likely

In [11]:
qry_nest = f"select {sql_nest} as pred from df_unif"
qry_line = f"select {sql_line} as pred from df_unif"
qry_slim = f"select {sql_slim} as pred from df_unif"

In [12]:
%%timeit -n 200 -r 200

con.sql(qry_nest)

361 μs ± 29.8 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
361 μs ± 29.8 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)


In [13]:
%%timeit -n 200 -r 200

con.sql(qry_line)

669 μs ± 11.1 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
669 μs ± 11.1 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)


In [14]:
%%timeit -n 200 -r 200

con.sql(qry_slim)

437 μs ± 8.65 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
437 μs ± 8.65 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)


#### Skewed Case

In this case, nodes are skewed, so we can see benefit from the linearized version ordering by node size 

In [15]:
qry_nest = f"select {sql_nest} as pred from df_skew"
qry_line = f"select {sql_line} as pred from df_skew"
qry_slim = f"select {sql_slim} as pred from df_skew"

In [16]:
%%timeit -n 200 -r 200

con.sql(qry_nest)

519 μs ± 34.7 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
519 μs ± 34.7 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)


In [17]:
%%timeit -n 200 -r 200

con.sql(qry_line)

970 μs ± 35.5 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
970 μs ± 35.5 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)


In [18]:
%%timeit -n 200 -r 200

con.sql(qry_slim)

696 μs ± 17.9 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
696 μs ± 17.9 μs per loop (mean ± std. dev. of 200 runs, 200 loops each)
